# Resource optimization workflow

Find the processes of the pipelines we use where better resource requests pay off, using **our own cluster runs as the
main data** and the **nf-core AWS tests as additional guidance**. The steps follow `docs/workflow.md`.

| Step | Section | |
|---|---|---|
| 0 | Data | AWS tables (`megatest_summary/`) and our traces (`data/<pipeline>/`) |
| 1 | Pipelines | choose below |
| 2 | Shortlist | four criteria: failures, runtime, inefficiency, correlation with input |
| 3 | Look closer | resource use against input size, per module |
| 4-5 | Decide, experiment | rules of thumb and the experiment protocol |
| 6 | Formula | not built yet |

Own runs also get the detailed views of the earlier notebook (priority ranking, burden per process, scaling across runs).

## 1. Setup and pipelines

In [ ]:
import subprocess
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "shortlist.py").exists():
    ROOT = ROOT.parent
SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

# ── Configuration ──────────────────────────────────────────────────────────────
PIPELINES = ["rnaseq", "ampliseq", "scrnaseq"]   # pipelines used at our institution, most used first
VIEW_PIPELINE = "rnaseq"                          # pipeline for the detailed views of our own runs (section 5)
AWS_DIR = ROOT / "megatest_summary"               # tables from the AWS tests (megatest_waste.py, input_sizes.py)
OWN_DIR = ROOT / "own_summary"                    # tables from our own traces (own_traces.py)
DATA_DIR = ROOT / "data"                          # our Nextflow traces: data/<pipeline>/*.txt (trace.raw = true)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)


def run_script(script, *args, quiet=False):
    """Run one of the repo scripts; print what it reports (nothing if quiet, unless it fails)."""
    r = subprocess.run([sys.executable, str(SCRIPTS / script), *map(str, args)], cwd=ROOT, capture_output=True, text=True)
    if not quiet:
        print(r.stdout[-3500:] or "")
    if r.returncode:
        print(r.stdout[-1500:], r.stderr[-2000:])


def header(text):
    """A bold line above an output, so it is clear which data it comes from."""
    display(Markdown(text))


FLAGS = {"failures": "F", "runtime": "R", "inefficient": "I", "input_dependent": "D"}


def flag_string(row):
    """Criteria met as letters, a dot where not met: F failures, R runtime, I inefficient, D input-dependent."""
    return " ".join(letter if row[name] else "·" for name, letter in FLAGS.items())


def compact(sl, per_pipeline=8):
    """The top rows per pipeline of a shortlist, with the criteria as one short column."""
    sl = sl.copy()
    sl["F R I D"] = sl.apply(flag_string, axis=1)
    cols = ["pipeline", "module", "criteria_met", "F R I D", "evidence", "tasks", "runs", "runtime_h", "reasons"]
    out = sl.groupby("pipeline", sort=False).head(per_pipeline)[cols]
    return out.style.hide(axis="index").format({"runtime_h": "{:.1f}"})


def newest(folder, pattern):
    files = list(Path(folder).glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

## 2. Data (step 0)

**AWS tables** come from `megatest_waste.py --all` and `input_sizes.py` (refresh only when needed, they take minutes and
need the network). **Our traces** are Nextflow trace files in `data/<pipeline>/`, written with `trace.raw = true`
(see section 7 for the fields). `own_traces.py` reduces them locally to the same per-task table; failed tasks are kept.

In [ ]:
header("**AWS tests** (nf-core megatests, guidance): saved tables")
print("\n".join(f.name for f in sorted(AWS_DIR.glob("*.parquet"))) or "no tables yet (scripts/megatest_waste.py --all)")

header("**Own runs** (our cluster traces, primary data): reduced locally from `data/<pipeline>/`")
run_script("own_traces.py", *PIPELINES)
own_path = OWN_DIR / "tasks__own.parquet"
own = pd.read_parquet(own_path) if own_path.exists() else None

aws_tasks = newest(AWS_DIR, "tasks__*.parquet")
aws_runs = (pd.read_parquet(aws_tasks, columns=["pipeline", "run"]).drop_duplicates().groupby("pipeline").size()
            if aws_tasks else pd.Series(dtype=int))
coverage = pd.DataFrame({
    "AWS runs": [int(aws_runs.get(p, 0)) for p in PIPELINES],
    "own runs": [int(own[own["pipeline"] == p]["run"].nunique()) if own is not None else 0 for p in PIPELINES],
    "own tasks": [int((own["pipeline"] == p).sum()) if own is not None else 0 for p in PIPELINES],
    "own failed tasks": [int(((own["pipeline"] == p) & (own["status"] == "FAILED")).sum()) if own is not None else 0 for p in PIPELINES],
}, index=pd.Index(PIPELINES, name="pipeline"))
header("**Coverage of the chosen pipelines**")
display(coverage)

## 3. Shortlist (step 2)

Four criteria, applied to every module of the chosen pipelines (thresholds are options of `scripts/shortlist.py`):
**failures** (failed or retried tasks), **runtime** (long or many tasks), **inefficient** (low CPU efficiency or large memory slack),
**input_dependent** (peak memory, runtime or CPU time correlate with `rchar` or the input size). No module is dropped for
having little data; those are marked `low data`. The research notes (`docs/tool_scaling_evidence.csv`) are joined where available.

Tables below read **`F R I D`**: one letter per criterion met (**F**ailures, **R**untime, **I**nefficient, **D** input-dependent),
a dot where it is not met. `evidence` is `low data` when a module has fewer than 10 tasks.

### 3.1 AWS tests (guidance)

In [ ]:
run_script("shortlist.py", *PIPELINES, quiet=True)
aws_sl = pd.read_csv(newest(AWS_DIR, f"shortlist__*__{'-'.join(sorted(PIPELINES))}.csv"))
header(f"**AWS tests**: {', '.join(PIPELINES)}; {int(aws_runs.reindex(PIPELINES).fillna(0).sum())} nf-core full-size test runs, "
       f"{len(aws_sl)} modules. Top 8 per pipeline by criteria met.")
display(compact(aws_sl))

### 3.2 Own runs (primary)

In [ ]:
own_sl = None
if own is not None:
    own_pipes = sorted(own["pipeline"].unique())
    run_script("shortlist.py", *own_pipes, "--data", own_path, "--out", OWN_DIR, quiet=True)
    own_sl = pd.read_csv(newest(OWN_DIR, f"shortlist__own__{'-'.join(own_pipes)}.csv"))
    runs = own.groupby("pipeline")["run"].nunique().to_dict()
    header(f"**Own runs**: {', '.join(f'{p} ({n} run' + ('s' if n != 1 else '') + ')' for p, n in runs.items())}; "
           f"{len(own_sl)} modules. Top 8 per pipeline by criteria met.")
    display(compact(own_sl))
    low = int((own_sl["evidence"] == "low data").sum())
    print(f"{low} of {len(own_sl)} modules have fewer than 10 tasks (low data). With one run per pipeline, rates and "
          "correlations are weak evidence; more runs and different sample sizes strengthen them.")
missing = [p for p in PIPELINES if own is None or p not in set(own["pipeline"])]
if missing:
    print(f"No own traces yet for: {', '.join(missing)} (only the AWS shortlist above exists for them).")

### 3.3 Own runs against AWS

Own runs are the main evidence, AWS the guidance. A module high on both is the safest candidate. High on the AWS side only
means it is worth checking in our runs; high on our side only means a setup specific to us (for example our requests).
Only pipelines with own traces are compared; `where` says which source lists the module.

In [ ]:
def _side(sl, name):
    s = sl[["pipeline", "module", "criteria_met", "evidence"]].copy()
    s[f"F R I D ({name})"] = sl.apply(flag_string, axis=1)
    return s.rename(columns={"criteria_met": f"criteria ({name})", "evidence": f"evidence ({name})"})


aws_part = _side(aws_sl, "AWS")
if own_sl is not None:
    compare = _side(own_sl, "own").merge(aws_part, on=["pipeline", "module"], how="outer")
    compare["where"] = ["both" if pd.notna(o) and pd.notna(a) else "own only" if pd.notna(o) else "AWS only"
                        for o, a in zip(compare["criteria (own)"], compare["criteria (AWS)"])]
    compare = compare.sort_values(["criteria (own)", "criteria (AWS)"], ascending=False, na_position="last")
    shown = compare[compare["pipeline"].isin(own_pipes)]
    order = ["pipeline", "module", "where", "criteria (own)", "F R I D (own)", "evidence (own)",
             "criteria (AWS)", "F R I D (AWS)", "evidence (AWS)"]
    header(f"**Own runs vs AWS** for {', '.join(own_pipes)}. Top 30 by own criteria met.")
    display(shown[order].head(30).style.hide(axis="index")
            .format({"criteria (own)": "{:.0f}", "criteria (AWS)": "{:.0f}"}, na_rep=""))
else:
    compare = aws_part
    print("No own traces yet: nothing to compare. See the AWS shortlist above.")

## 4. Look closer (step 3)

Peak memory, runtime and CPU time against input size, one task per dot (log-log, power-law fit). The panel titles give
the Spearman correlation (rho, averaged over pipelines) and the exponent b of y = a·x^b (near 0: flat, near 1: proportional).
Choose the data source and the x axis: real input size exists only for AWS runs whose samplesheet could be resolved,
`rchar` (bytes read by the task) is available for nearly all tasks and for our own runs.

In [ ]:
from plot_scaling import load_points, make_figure

SOURCES = {"AWS tests": (newest(AWS_DIR, "tasks__*.parquet"), AWS_DIR)}
if own_path.exists():
    SOURCES["Own runs"] = (own_path, OWN_DIR)
_points = {}

top = compare[compare["pipeline"].isin(PIPELINES)]["module"].drop_duplicates().head(40).tolist()
source_w = widgets.Dropdown(options=list(SOURCES), description="Data:")
x_w = widgets.Dropdown(options=["input", "rchar"], value="rchar", description="x axis:")
module_w = widgets.Combobox(options=top, value=top[0] if top else "", description="Module:", ensure_option=False)
plot_out = widgets.Output()


def draw(_=None):
    plot_out.clear_output(wait=True)
    with plot_out:
        key = (source_w.value, x_w.value)
        if key not in _points:
            try:
                data, out = SOURCES[source_w.value]
                _points[key] = load_points(data, out, x_w.value)[:2]
            except SystemExit as e:
                print(e)
                return
        t, xlabel = _points[key]
        d = t[t["module"] == module_w.value.upper()]
        if d.empty:
            print(f"No tasks for {module_w.value!r} with x = {x_w.value} in {source_w.value}")
            return
        fig = make_figure(module_w.value.upper(), d, xlabel)
        fig.layout.title.text = f"[{source_w.value}] " + fig.layout.title.text
        fig.show(renderer="plotly_mimetype")


for w in (source_w, x_w, module_w):
    w.observe(draw, names="value")
display(widgets.HBox([source_w, x_w, module_w]), plot_out)
draw()

## 5. Detailed views of our own runs

Carried over from the earlier notebook: the **optimization priority** ranking (impact × inefficiency × variability, with
adjustable weights), the **resource burden** per process within a run, and the **scaling explorer** that compares
consistent processes across runs (useful once the same pipeline has been run with inputs of different sizes).
Traces are loaded from `data/<VIEW_PIPELINE>/` (folder name contains the pipeline name). These views use only completed
or cached tasks; failures are covered by the shortlist above.

In [ ]:
from ipywidgets import Dropdown

from workflow_helpers import (
    create_optimization_priority_plot,
    create_process_composition_plot,
    create_process_scaling_explorer,
    load_pipeline_traces,
)

header(f"**Own runs only** (`data/{VIEW_PIPELINE}/`); no AWS data is used in this section.")
run_summary_df, process_df, trace_files, labels = load_pipeline_traces(VIEW_PIPELINE, DATA_DIR)

### 5.1 Optimization priority

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# Controls
# -----------------------------
run_label_widget = widgets.Dropdown(
    options=labels,
    description="Run:",
    style={"description_width": "initial"},
)

top_n_widget = widgets.IntSlider(
    value=20,
    min=5,
    max=30,
    step=1,
    description="Top N:",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Impact sliders
impact_cpu_widget = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU cost",
    style={"description_width": "initial"},
    continuous_update=False,
)
impact_mem_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory cost",
    style={"description_width": "initial"},
    continuous_update=False,
)
impact_time_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Time cost",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Inefficiency sliders
ineff_cpu_widget = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)
ineff_mem_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)
ineff_time_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Time ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Variability sliders
var_runtime_widget = widgets.FloatSlider(
    value=0.6,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Runtime CV",
    style={"description_width": "initial"},
    continuous_update=False,
)
var_cpu_widget = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU CV",
    style={"description_width": "initial"},
    continuous_update=False,
)
var_mem_widget = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory CV",
    style={"description_width": "initial"},
    continuous_update=False,
)

# -----------------------------
# Plot function
# -----------------------------
output = widgets.Output()


def show_optimization_dashboard(
    run_label,
    top_n,
    impact_cpu,
    impact_mem,
    impact_time,
    ineff_cpu,
    ineff_mem,
    ineff_time,
    var_runtime,
    var_cpu,
    var_mem,
):
    idx = labels.index(run_label)
    stem = trace_files[idx].stem
    metrics = process_df[process_df["filename"] == stem].copy()

    impact_weights = {
        "cpu_cost": impact_cpu,
        "mem_cost": impact_mem,
        "time_cost": impact_time,
    }
    inefficiency_weights = {
        "cpu_ineff": ineff_cpu,
        "mem_ineff": ineff_mem,
        "time_ineff": ineff_time,
    }
    variability_weights = {
        "runtime_cv": var_runtime,
        "cpu_cv": var_cpu,
        "mem_cv": var_mem,
    }

    fig = create_optimization_priority_plot(
        metrics=metrics,
        impact_weights=impact_weights,
        inefficiency_weights=inefficiency_weights,
        variability_weights=variability_weights,
        top_n=top_n,
    )

    with output:
        clear_output(wait=True)
        fig.show()


# -----------------------------
# Grouped layout in two rows
# -----------------------------
general_box = widgets.VBox(
    [
        widgets.HTML("<h4 style='margin:0;'>General</h4>"),
        widgets.HTML(
            "<div style='font-size:12px;color:#666;'>"
            "Weights are normalized within each section."
            "</div>"
        ),
        run_label_widget,
        top_n_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px 12px",
        margin="6px 0 10px 0",
        width="100%",
    ),
)

shared_box_layout = widgets.Layout(
    border="1px solid #ddd",
    padding="10px 12px",
    margin="0 8px 0 0",
    width="33%",
)

impact_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Impact</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of compute cost components"
            "</div>"
        ),
        impact_cpu_widget,
        impact_mem_widget,
        impact_time_widget,
    ],
    layout=shared_box_layout,
)

ineff_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Inefficiency</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of resource inefficiencies"
            "</div>"
        ),
        ineff_cpu_widget,
        ineff_mem_widget,
        ineff_time_widget,
    ],
    layout=shared_box_layout,
)

var_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Variability</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of input-dependent variability"
            "</div>"
        ),
        var_runtime_widget,
        var_cpu_widget,
        var_mem_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px 12px",
        margin="0",
        width="33%",
    ),
)

slider_row = widgets.HBox(
    [impact_box, ineff_box, var_box],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start",
        justify_content="space-between",
    ),
)

controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3 style='margin:0 0 8px 0;'>Optimization Priority Dashboard</h3>"
        ),
        general_box,
        slider_row,
    ]
)

# -----------------------------
# Bind widgets to function
# -----------------------------
interactive = widgets.interactive_output(
    show_optimization_dashboard,
    {
        "run_label": run_label_widget,
        "top_n": top_n_widget,
        "impact_cpu": impact_cpu_widget,
        "impact_mem": impact_mem_widget,
        "impact_time": impact_time_widget,
        "ineff_cpu": ineff_cpu_widget,
        "ineff_mem": ineff_mem_widget,
        "ineff_time": ineff_time_widget,
        "var_runtime": var_runtime_widget,
        "var_cpu": var_cpu_widget,
        "var_mem": var_mem_widget,
    },
)

display(controls, output, interactive)

### 5.2 Resource burden per process

In [ ]:
composition_run_widget = Dropdown(options=labels, description="Run:")
composition_sort_widget = Dropdown(
    options=[
        ("CPU", "cpu_h"),
        ("Memory", "memory_h"),
        ("Runtime", "total_runtime_hours"),
        ("I/O", "io_gb"),
    ],
    description="Sort by:",
)
composition_output = widgets.Output()


def show_process_composition_dashboard(run_label, sort_by):
    idx = labels.index(run_label)
    stem = trace_files[idx].stem

    pp = process_df[process_df["filename"] == stem].copy()

    fig = create_process_composition_plot(pp, run_label, sort_by)
    fig.update_layout(width=1100, height=400)

    composition_output.clear_output(wait=True)
    with composition_output:
        fig.show(renderer="plotly_mimetype")


_ = widgets.interactive_output(
    show_process_composition_dashboard,
    {
        "run_label": composition_run_widget,
        "sort_by": composition_sort_widget,
    },
)

display(
    widgets.VBox(
        [
            widgets.HBox([composition_run_widget, composition_sort_widget]),
            composition_output,
        ]
    )
)

### 5.3 Scaling across runs

In [ ]:
TOP_PROCESSES_PER_RESOURCE = 15

fig = create_process_scaling_explorer(
    process_df,
    run_summary_df,
    top_n=TOP_PROCESSES_PER_RESOURCE,
)
if fig is None:
    print("No consistent processes found")
else:
    fig.show(renderer="plotly_mimetype")

## 6. Decide per module (step 4)

| Finding | Next step |
|---|---|
| Resources correlate with input, input range wide (>= 10 distinct sizes, max/min >= 10) | fit a request formula |
| Resources correlate with input, range narrow | run an experiment with a wider input range |
| Flat resource use, request far above use | try a smaller fixed request, confirm on more data |
| High failure rate | find the cause first (exit 137/140/143: memory or walltime) |
| Research says a reference or a setting drives the resource (e.g. STAR memory by genome size) | base the request on that driver |

The `next_step` column of the shortlist applies these rules.

## 7. Experiments on our own data (step 5)

For modules that need a wider input range or more data. Use one tool version and thread count, at least 5 input sizes over one to two
orders of magnitude (for example read subsets at 5, 10, 25, 50, 100 %), 3 repeats each. Record the trace with these fields
(raw values, no units), then put the files into `data/<pipeline>/` and rerun sections 2-3:

```groovy
trace {
  enabled = true
  raw = true
  fields = "process,tag,task_id,name,status,exit,attempt,cpus,memory,time,realtime,%cpu,peak_rss,rchar,wchar,read_bytes,write_bytes"
}
```

## 8. Formula (step 6, not built yet)

Per module: runtime and memory against input size (power law, or linear with intercept; flat plus margin where there is no
relation), validated by leaving one pipeline or run out; a safety margin so that 95-99 % of tasks fit; compared with the
current request. Suggested first modules from the AWS analysis: `PICARD_MARKDUPLICATES`, `TRIMGALORE`, `FASTQC`.